In [32]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import torch.nn.functional as F



In [ ]:
"""


Chatamari
Chhoila
Dalbhat
Dhindo
Gundruk
Kheer
Momo
Sekuwa
Selroti

"""

In [10]:
categories = [
    "Chatamari",
    "Chhoila",
    "Dalbhat",
    "Dhindo",
    "Gundruk",
    "Kheer",
    "Momo",
    "Sekuwa",
    "Selroti"
]


In [15]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")





In [4]:
device

device(type='cpu')

In [16]:

class SimpleResNetWithConv(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
       
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
       
        for param in resnet.parameters():
            param.requires_grad = False
        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) 
        self.conv = nn.Conv2d(in_channels=2048, out_channels=1000, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=1000, out_channels=512, kernel_size=3, padding=1)
        self.my_relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.my_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear_1 = nn.Linear(512, 125)
        self.linear_2 = nn.Linear(125, num_classes)
    
    def forward(self, x):
       
        x = self.backbone(x)
        
        x = self.my_relu(self.conv(x))
        x = self.my_relu(self.conv2(x))
        x = self.my_pool(x)
        x = torch.flatten(x, 1)
        x = self.linear_1(x)
        x = self.dropout(x)
        return self.linear_2(x)

In [17]:


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [6]:
model = SimpleResNetWithConv(num_classes=9)

In [18]:
model.load_state_dict(torch.load("/Users/kbshal/mero_space/projects/finetune_nepali_images/best_nepali_food_model.pth", map_location=device))
model.to(device)
model.eval()

SimpleResNetWithConv(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
  

In [33]:


def predict_image(image_path, top_k=5):
    # weights.transforms() = the EXACT resize/crop/normalize this model expects
    preprocess = train_transforms

    img = Image.open(image_path).convert("RGB")
    batch = preprocess(img).unsqueeze(0)   # [3,224,224] -> [1,3,224,224]

    with torch.no_grad():                  # inference, not training -- no gradients needed
        logits = model(batch)              # [1, 1000] raw scores
        probs = F.softmax(logits[0], dim=0)

    top_probs, top_idx = torch.topk(probs, top_k)
    return [(categories[i], p.item()) for p, i in zip(top_probs, top_idx)]

In [34]:
predict_image("choila.png")

[('Chhoila', 0.6492974162101746),
 ('Sekuwa', 0.34948021173477173),
 ('Gundruk', 0.0010225202422589064),
 ('Dhindo', 0.00010693582589738071),
 ('Chatamari', 6.60953446640633e-05)]

In [35]:
predict_image("kheer.png")

[('Kheer', 1.0),
 ('Dalbhat', 3.4576785878925875e-10),
 ('Chatamari', 5.988076701157752e-11),
 ('Momo', 2.743262214610631e-11),
 ('Sekuwa', 1.053671299516834e-11)]

In [46]:
predict_image("selroti.png")


[('Selroti', 0.9999696016311646),
 ('Sekuwa', 2.9175131203373894e-05),
 ('Dhindo', 9.469994779465196e-07),
 ('Momo', 2.934817473487783e-07),
 ('Kheer', 1.224557255596892e-08)]